# **CHAPTER 4: Text Classification**

Due to the broad field of text classification, we will discuss several techniques and use them to explore the field of language models:
<ul>
    <li><b>Text Classification with Representation Models</b> demonstrates the flexibility of non-generative models for classification.</li>
    <li><b>Text Classification with Generative Models</b> is an introduction to generative language models as most of them can be used for classification.</li>
</ul>

### **The sentiment of Movie Reviews**

In this section, we will use the **well-known** *rotten_tomatoes* datasets to train and evaluate our models. It contains **5,331 positive and 5,331 negative movie reviews** from Rotten Tomatoes.

In [18]:
!pip install -q datasets
!pip install -q transformers numpy pandas matplotlib
!pip install -q sentence-transformers
!pip install -q scikit-learn

In [4]:
from datasets import load_dataset

data = load_dataset("rotten_tomatoes")
data

README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [5]:
data['train'][0, -1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

## **Task Classification with Representation Models**

As we explored in the previous chapter, these models are created by **fine-tuning** a foundation model, like BERT, on a specific downstream task as illustrated:

<img src="../image/two_flavor_models.jpg" style="display:block; margin:0 auto">

A **task-specific model** is a representation model, such as BERT, trained for a specific task, like **sentiment analysis**. An embedding model generates **general-purpose** embeddings that can be used for a variety of tasks not limited to classification, like **semantic search**.

And in this section, we keep both models ***frozen*** (non-trainable) and only use their output as shown in the picture below:

<img src="../image/frozen_models.jpg" style="display:block; margin:0 auto">

## **Model Selection**

**BERT**, a well-known **encoder-only** architecture, is a popular choice for creating task-specific and embedding models. While generative models, like the **GPT family**, are incredible models, encoder-only models similarly excel in task-specific use cases and tend to be significantly smaller in size.

<ul>
    <li><b>BERT</b>: We use <b>BERT</b> when we need a model to do <b>Classification Tasks, Understanding Tasks.</b> It requires some crucial skill such as <b>text similarity, semantic search, document ranking, paraphrase detection.t</b></li>
    <li>
    <b>GPT</b>: We use <b>GPT family</b> when we need models to do <b>Generation Tasks, Few-shot/Zero-shot Learning</b>.
    </li>
</ul>

When selecting models to generate embeddings from, <a href="https://oreil.ly/mUVXD"><b>the MTEB leaderboard</b></a> is a great place to start. It contains open and closed source models benchmarked across several tasks.

In [6]:
from transformers import pipeline

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device="cuda:0"
)

pipe

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [7]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

y_pred = []

isPrinted = False
for output in tqdm(pipe(KeyDataset(data['test'], "text")), total=len(data['test'])):
    if not isPrinted:
        print(output)
        isPrinted = True
        
    nega_score = output[0]['score']
    posi_score = output[2]['score']

    # argmax() returns the index of the element that has the maximum value in the input list
    # if neg_score > posi_score -> return 0
    # else return 1
    assignment = np.argmax([nega_score, posi_score])
    y_pred.append(assignment)
y_pred


  0%|          | 5/1066 [00:00<02:22,  7.43it/s]

[{'label': 'negative', 'score': 0.00516123790293932}, {'label': 'neutral', 'score': 0.040233541280031204}, {'label': 'positive', 'score': 0.9546052813529968}]



100%|██████████| 1066/1066 [00:10<00:00, 103.20it/s]


[1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,


In [8]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
    performance = classification_report(
        y_true, y_pred,
        target_names = ["Negative Review", "Positive Review"]
    )

    print(performance)

In [9]:
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.76      0.88      0.81       533
Positive Review       0.86      0.72      0.78       533

       accuracy                           0.80      1066
      macro avg       0.81      0.80      0.80      1066
   weighted avg       0.81      0.80      0.80      1066



<img src="https://raw.githubusercontent.com/khang1108/learning_ai/refs/heads/main/image/types_predictions.jpg" style="display:block; margin:0 auto">

Using the confusion matrix, we can derive several formulas to describe the quality of the model. 
- **Precision** measures how many of the items found are relevant, which indicates the accuracy of the relevant results. It means that all of the items predicted as **POSITIVE**, how many are truly positive? **Precision** will be more important if missing a **True Positive** - getting **False Positive** causes a big problem such as a **multi-billion contract**. The higher the precision, the more accurate the identified points are.

$$
Precision = \frac{TruePositive}{TruePositive + FalsePositive}
$$

- **Recall** refers to how many relevant classes were found, which indicates its ability to find all relevant results

$$
Recall = \frac{TruePositive}{TruePositive + FalseNegative}
$$


- **Accuracy** refers to how many correct predictions the model makes out of all predictions, which indicates the **OVERALL CORRECTNESS** of the model.

$$
Accuracy = \frac{TruePositive + TrueNegative}{TruePositive + TrueNegative + FalsePositive + FalseNegative}
$$


- **F1 score** balances both precision and recall to create a model's overall performance.

$$
F1= 2 \frac{Precision * Recall}{Precision + Recall}
$$


## **Supervised Classification**

Instead of directly using the **representation model for classification**, we will use an **embedding model for generating features**. Those features can then be fed into a **classifier**, thereby creating a two-step approach.

A major benefit of this separation is that we do **NOT** need to fine-tune our embedding model, which can be **COSTLY**. In contrast, we can train a classifier, like a logistic regression, on the CPU instead.

In the first step, we convert our textual input to embeddings using the embedding model. Note that this model is similarly kept **frozen** and is not updated during the training process.

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

train_embeddings = model.encode(data['train']['text'], show_progress_bar=True)
test_embeddings = model.encode(data['test']['text'], show_progress_bar=True)

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

In [12]:
train_embeddings.shape

(8530, 768)

In [13]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data['train']['label'])

y_pred = clf.predict(test_embeddings)
evaluate_performance(data['test']['label'], y_pred)


                 precision    recall  f1-score   support

Negative Review       0.85      0.86      0.85       533
Positive Review       0.86      0.85      0.85       533

       accuracy                           0.85      1066
      macro avg       0.85      0.85      0.85      1066
   weighted avg       0.85      0.85      0.85      1066



And finally, by training a classifier on top of our embeddings, we managed to get an **F1 score of 0.85**. This demonstrates the possibiliets of training a lightweight classifier while keeping the underlying embedding model frozen.

## **What if we do not have labelled data?**

In practice, getting labelled data is a resource-intensive task that can require significant human labor. So that is it actually worthwhile to collect these labels?

To test this, we can perform **zero-shot** classification, where we have no labelled data to explore whether the task seems feasible. **Zero-shot** classification attempts to predict the labels of input text even though it was not trained on them before. 

To perform **zero-shot** classification with embeddings, there is a neat trick that we can use. We can describe our labels based on what they should represent. For example, a negative label for movie reviews can be described as **"This is a negative movie review."**. By describing and embedding the labels and documents, we have data that we can work with.

In [14]:
label_embeddings = model.encode(['A negative review', 'A positive review'])
label_embeddings

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

array([[ 0.04459879, -0.03335255,  0.02032453, ...,  0.00385526,
         0.0405459 , -0.01143497],
       [ 0.04103614, -0.02912743, -0.00471624, ...,  0.02639272,
         0.033573  , -0.04018765]], dtype=float32)

In [15]:
label_embeddings.shape

(2, 768)

<img src="https://raw.githubusercontent.com/khang1108/learning_ai/refs/heads/main/image/value_embeddings.jpg" style="display:block; margin:0 auto">

As we can see in the picture, each input text has a unique value on the plane. To calculate the similarity of these input texts, we can use **cosine similarity**. Between each label input and the input text is an angle called $\theta$; using this angle, we can formulate an equation to calculate the similarity.

<img src="https://raw.githubusercontent.com/khang1108/learning_ai/refs/heads/main/image/cosine_value.jpg" style="display:block; margin:0 auto">

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

In [22]:
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.78      0.77      0.78       533
Positive Review       0.77      0.79      0.78       533

       accuracy                           0.78      1066
      macro avg       0.78      0.78      0.78      1066
   weighted avg       0.78      0.78      0.78      1066



# **Text Classification with Generative Models**

As we have seen before, using **an embedding model** returns an impressive **F1-score**, which shows how versatile and useful embeddings are. And moving to the next section, we will dive into **Generative Models** for Text Classification.

**Generative Models** are decoder-only models, such as OpenAI's GPT models, that work a bit differently from what we have done so far. These models take as input some text and generative text and are thereby aptly named **sequence-to-sequence** models. These models are generally trained on a wide variety of tasks and usually do not perform your use case out of the box. For instance, if we give a generative model a movie review without any context, it has no idea what to do with it.

Instead, we need to help it understand the context and guide it toward the answers that we are looking for.

## **Using the Text-to-Text Transfer Transformer**

An interesting family of models that leverage this architecture is the **Text-to-Text** Transformer or **T5 model**. Its architecture is similar to the original Transformer, where **12** encoders and **12** decoders are stacked together.

<img src="https://raw.githubusercontent.com/khang1108/learning_ai/refs/heads/main/image/T5_architecture.jpg" style="display:block; margin:0 auto">

With this architecture, these models were first **pretrained** using **masked language modeling**. In the first step of training, instead of masking individual tokens, sets of tokens (or token spans) were masked during pretraining.

The second step of training, namely **fine-tuning** the base model, is where the real magic happens. Instead of fine-tuning the model for one specific task, each task is converted to a **sequence-to-sequence** task and trained simultaneously.

In [23]:
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device="cuda:0"
)

prompt = "Is the following sentence positive or negative?"
data = data.map(lambda example: {"t5": prompt + example['text']})
data

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

Device set to use cuda:0


Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
})

In [26]:
data['train'][0]['t5']

'Is the following sentence positive or negative?the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .'

In [27]:
y_pred = []

for output in tqdm(pipe(KeyDataset(data['test'], 't5')), total=len(data['test'])):
    text = output[0]['generated_text']
    y_pred.append(0 if text == "negative" else 1)

evaluate_performance(data['test']['label'], y_pred)

100%|██████████| 1066/1066 [00:50<00:00, 21.08it/s]

                 precision    recall  f1-score   support

Negative Review       0.83      0.84      0.83       533
Positive Review       0.84      0.83      0.83       533

       accuracy                           0.83      1066
      macro avg       0.83      0.83      0.83      1066
   weighted avg       0.83      0.83      0.83      1066



In [29]:
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device="cuda:0"
)

y_pred = []

for output in tqdm(pipe(KeyDataset(data['test'], 't5')), total=len(data['test'])):
    text = output[0]['generated_text']
    y_pred.append(0 if text == "negative" else 1)

evaluate_performance(data['test']['label'], y_pred)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
100%|██████████| 1066/1066 [01:01<00:00, 17.29it/s]

                 precision    recall  f1-score   support

Negative Review       0.86      0.92      0.89       533
Positive Review       0.91      0.85      0.88       533

       accuracy                           0.88      1066
      macro avg       0.89      0.88      0.88      1066
   weighted avg       0.89      0.88      0.88      1066



As we can see, selecting a version of the model **Flan-T5** is very important, which directly affects the **weighted average F1-score**.